# Classificação e Rastreio da Depressão com Support Vector Machines (SVM)

Neste notebook, aplicamos o algoritmo **LinearSVC** para a classificação binária do NHIS 2025. O SVM busca o hiperplano de separação de máxima margem entre as classes, oferecendo uma abordagem matemática baseada em fronteira linear para comparar com os modelos de árvore (Random Forest e XGBoost).

O objetivo é rastrear a depressão **sem utilizar sintomas de transtornos mentais como preditores**: todas as variáveis de depressão e de ansiedade são removidas de $X$, restando apenas variáveis sociodemográficas, de saúde física, de acesso a serviços de saúde e de contexto social.

## 1. Importação de Bibliotecas e Configuração dos Parâmetros Iniciais
Nesta etapa, carregamos os módulos essenciais do `scikit-learn` (incluindo o algoritmo `LinearSVC` e utilitários de pré-processamento) e lemos a base limpa diretamente de `01 Preparação dos dados/dados/limpos/adult25_limpo.csv`, por meio de um caminho relativo ao notebook (sem cópias locais do arquivo). Em seguida, definimos as colunas removidas de $X$ para evitar *data leakage*: os itens, o histórico, a medicação, a frequência, a intensidade e a severidade de **depressão (PHQ)** e de **ansiedade (GAD-7)**, de modo que o modelo não tenha acesso a sintomas de transtornos mentais. Por fim, transformamos o target original multiclasse em uma abordagem **binária** (`0`: Sem Depressão, `1`: Com Depressão).

In [8]:
import numpy as np
import pandas as pd
import time
from pathlib import Path
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Carregar a base limpa e aplicar a transformação binária
df = pd.read_csv(Path('../../01 Preparação dos dados/dados/limpos') / 'adult25_limpo.csv').set_index('ID_DOMICILIO')
df = df[~df['SEVERIDADE_DEPRESSAO_PHQ'].isin([-1, 5])].copy()
df['DEPRESSAO_PHQ_SIMPLIFICADO'] = (df['SEVERIDADE_DEPRESSAO_PHQ'] != 1).astype(int)

TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
PESO_COLUNA = 'PESO_AMOSTRAL'

COLUNAS_SINTOMAS = [
    # Depressão (PHQ): itens, histórico, medicação, frequência, intensidade e severidade (target original)
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ',
    # Ansiedade (GAD-7): histórico, medicação, frequência, intensidade, itens e severidade
    'HISTORICO_TRANSTORNO_ANSIEDADE', 'MED_ANSIEDADE',
    'FREQ_ANSIEDADE', 'INTENSIDADE_ANSIEDADE',
    'FREQ_ANSIEDADE_2SEM', 'FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM',
    'FREQ_PREOCUPACAO_DESNECESSARIA_2SEM', 'FREQ_DIFICULDADE_DESCANSAR_2SEM',
    'FREQ_AGITACAO_INQUIETA', 'FREQ_IRRITABILIDADE_2SEM',
    'FREQ_MEDO_2SEM', 'SEVERIDADE_ANSIEDADE'
]



## 2. Seleção de Features e Filtragem Estrita de Respostas
Isolamos as variáveis preditoras ($X$), o target binário ($y$) e o peso amostral (`PESO_AMOSTRAL`). Aplicamos o filtro estrito (Cenário B) para remover registros com códigos de não-resposta (`-1`) nas variáveis preditoras, garantindo a integridade metodológica e evitando ruídos no treinamento. Como as colunas de depressão e de ansiedade já foram descartadas de $X$, o filtro considera apenas as variáveis preditoras restantes.

In [9]:
# 2. Isolar X, y e Pesos (Apenas registros completos)
y_bin = df[TARGET_BIN]
weights = df[PESO_COLUNA]
X_base = df.drop(columns=[col for col in [TARGET_BIN, PESO_COLUNA] + COLUNAS_SINTOMAS if col in df.columns])

mascara_completos = (X_base != -1).all(axis=1)
X_bin = X_base[mascara_completos].copy()
y_bin = y_bin[mascara_completos].copy()
w_bin = weights[mascara_completos].copy()



## 3. Divisão Estratificada do Conjunto de Dados (Train / Test Split)
Dividimos a base de dados estrita em **80% para treino** e **20% para teste**. O uso do parâmetro `stratify=y_bin` garante que a proporção exata entre pessoas com e sem depressão seja preservada em ambos os subconjuntos, além de transportar o peso de expansão populacional (`w_tr` e `w_te`).

In [10]:
# 3. Split Estratificado
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
    X_bin, y_bin, w_bin, test_size=0.2, random_state=42, stratify=y_bin
)



## 4. Padronização Numérica e Codificação Categórica (Preprocessing)
Construímos o pipeline de transformação de dados (`ColumnTransformer`):
* **Normalização (`StandardScaler`):** Aplicada às variáveis contínuas (como `RAZAO_RENDA_POBREZA`) para colocar os valores na mesma escala (essencial para o cálculo de margem do SVM).
* **One-Hot Encoding (`OneHotEncoder`):** Transforma as variáveis categóricas sociodemográficas em vetores binários, descartando a primeira categoria para evitar multicolinearidade.

In [11]:
# 4. Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA']
cat_cols = [c for c in X_tr.columns if c not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_tr_proc = preprocessor.fit_transform(X_tr)
X_te_proc = preprocessor.transform(X_te)



## 5. Definição do Espaço de Busca de Hiperparâmetros
Configuramos a grade de hiperparâmetros para otimização do `LinearSVC`:
* **`C` (Parâmetro de Regularização):** Controla o *trade-off* entre obter uma margem mais larga e minimizar os erros de classificação.
* **`class_weight`:** Testamos o ajuste automático de pesos (`balanced` vs `None`) para lidar com o desbalanceamento das classes.
* **`tol` (Tolerância do Critério de Parada):** Controla a precisão de convergência do algoritmo.

In [12]:
# 5. Espaço de Busca de Hiperparâmetros para o LinearSVC
param_dist_svm = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0],
    'class_weight': ['balanced', None],
    'tol': [1e-4, 1e-3, 1e-2]
}



## 6. Otimização do SVM com Validação Cruzada (RandomizedSearchCV)
Executamos a busca estocástica com 10 combinações em 3 dobras de validação cruzada (`cv=3`), otimizando diretamente a métrica de $F1\text{-score}$. O algoritmo treina a fronteira de separação considerando o peso amostral populacional (`sample_weight`).

In [13]:
# 6. Otimização com RandomizedSearchCV
svm_base = LinearSVC(random_state=42, max_iter=3000, dual=False)

random_search_svm = RandomizedSearchCV(
    estimator=svm_base,
    param_distributions=param_dist_svm,
    n_iter=10,
    scoring='f1',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("Iniciando otimização do SVM (LinearSVC)...")
inicio = time.time()
random_search_svm.fit(X_tr_proc, y_tr, sample_weight=w_tr)
tempo_total = time.time() - inicio

svm_otimizado = random_search_svm.best_estimator_

print(f"\n✓ Otimização concluída em {tempo_total:.2f}s")
print("\n--- MELHORES HIPERPARÂMETROS ENCONTRADOS (SVM) ---")
for param, val in random_search_svm.best_params_.items():
    print(f"{param}: {val}")



Iniciando otimização do SVM (LinearSVC)...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

✓ Otimização concluída em 62.04s

--- MELHORES HIPERPARÂMETROS ENCONTRADOS (SVM) ---
tol: 0.01
class_weight: balanced
C: 0.01


## 7. Avaliação e Projeção Populacional do Modelo SVM
Avaliamos o modelo final no conjunto de teste independente:
1. **Função de Decisão (`decision_function`):** Calcula a distância das instâncias até o hiperplano de separação para computar o score ROC-AUC.
2. **Matrizes de Confusão Ponderadas:** Exibimos os números absolutos e percentuais de Verdadeiros Negativos (TN), Falsos Positivos (FP), Falsos Negativos (FN) e Verdadeiros Positivos (TP) tanto para a amostra local quanto para a projeção nacional expandida nos EUA.

In [14]:
# 7. Avaliação no Conjunto de Teste
y_pred_svm = svm_otimizado.predict(X_te_proc)
y_score_svm = svm_otimizado.decision_function(X_te_proc)

print("\n--- RELATÓRIO DE CLASSIFICAÇÃO SVM (PONDERADO) ---")
print(classification_report(y_te, y_pred_svm, sample_weight=w_te, zero_division=0))

# Quadrantes da Matriz de Confusão
cm_amostra = confusion_matrix(y_te, y_pred_svm)
tn, fp, fn, tp = cm_amostra.ravel()
total_amostra = len(y_te)

cm_pop = confusion_matrix(y_te, y_pred_svm, sample_weight=w_te)
tn_w, fp_w, fn_w, tp_w = cm_pop.ravel()
total_pop = w_te.sum()

print("--- DETALHAMENTO DA MATRIZ DE CONFUSÃO DA AMOSTRA (Linhas de Teste) ---")
print(f"Verdadeiros Negativos (TN): {tn:5d} | {tn/total_amostra*100:6.2f}% (Sem Depressão - Correto)")
print(f"Falsos Positivos      (FP): {fp:5d} | {fp/total_amostra*100:6.2f}% (Alarme Falso)")
print(f"Falsos Negativos      (FN): {fn:5d} | {fn/total_amostra*100:6.2f}% (Subdiagnóstico)")
print(f"Verdadeiros Positivos (TP): {tp:5d} | {tp/total_amostra*100:6.2f}% (Com Depressão - Capturado)")

print("\n--- DETALHAMENTO POPULACIONAL EXPANDIDO (Projeção Nacional) ---")
print(f"Verdadeiros Negativos (TN): {tn_w:11,.0f} | {tn_w/total_pop*100:6.2f}%")
print(f"Falsos Positivos      (FP): {fp_w:11,.0f} | {fp_w/total_pop*100:6.2f}%")
print(f"Falsos Negativos      (FN): {fn_w:11,.0f} | {fn_w/total_pop*100:6.2f}%")
print(f"Verdadeiros Positivos (TP): {tp_w:11,.0f} | {tp_w/total_pop*100:6.2f}%")

print(f"\nROC-AUC Score (SVM): {roc_auc_score(y_te, y_score_svm, sample_weight=w_te):.4f}")


--- RELATÓRIO DE CLASSIFICAÇÃO SVM (PONDERADO) ---
              precision    recall  f1-score   support

           0       0.93      0.83      0.88 38513852.34799991
           1       0.53      0.75      0.62 9894699.189000001

    accuracy                           0.81 48408551.53699991
   macro avg       0.73      0.79      0.75 48408551.53699991
weighted avg       0.85      0.81      0.82 48408551.53699991

--- DETALHAMENTO DA MATRIZ DE CONFUSÃO DA AMOSTRA (Linhas de Teste) ---
Verdadeiros Negativos (TN):  2876 |  64.67% (Sem Depressão - Correto)
Falsos Positivos      (FP):   626 |  14.08% (Alarme Falso)
Falsos Negativos      (FN):   223 |   5.01% (Subdiagnóstico)
Verdadeiros Positivos (TP):   722 |  16.24% (Com Depressão - Capturado)

--- DETALHAMENTO POPULACIONAL EXPANDIDO (Projeção Nacional) ---
Verdadeiros Negativos (TN):  31,914,296 |  65.93%
Falsos Positivos      (FP):   6,599,556 |  13.63%
Falsos Negativos      (FN):   2,438,292 |   5.04%
Verdadeiros Positivos (TP):   7,

### Análise Crítica dos Resultados: Support Vector Machines (LinearSVC)

**Cenário avaliado:** previsão da depressão **sem sintomas de transtornos mentais em $X$**. Foram removidos os itens, o histórico, a medicação, a frequência, a intensidade e a severidade de depressão (PHQ) e de ansiedade (GAD-7), ou seja, as 25 colunas de `COLUNAS_SINTOMAS`. O modelo usa apenas variáveis sociodemográficas, de saúde física, de acesso a serviços de saúde e de contexto social.

* **Capacidade Discriminativa (ROC-AUC = 0.8703):** O score ROC-AUC obtido a partir da função de decisão mostra que, mesmo sem sintomas de saúde mental, as variáveis restantes ainda oferecem uma separação linear consistente. Na versão anterior, com as variáveis de ansiedade ainda em $X$, o ROC-AUC era 0.9109; a queda de cerca de 0.04 indica que parte do desempenho anterior vinha dessas variáveis (ansiedade e depressão são fortemente comórbidas), e não apenas do perfil sociodemográfico.
* **Sensibilidade Elevada (Recall = 75,4%):** A busca de hiperparâmetros selecionou `class_weight: balanced` (com `C: 0.01`), o que desloca a margem do hiperplano em favor da classe minoritária. Na projeção populacional, o SVM identificou corretamente **7.456.407 indivíduos com depressão** (15,40% da população) e deixou de detectar **2.438.292** (falsos negativos = 5,04% da população, ou 24,6% dos casos reais).
* **Custo em Precisão (Precisão = 53,0%):** O ganho de sensibilidade vem acompanhado de mais alarmes falsos: os falsos positivos somam **6.599.556 pessoas** (13,63% da população), e pouco mais da metade (53 em cada 100) das pessoas classificadas como positivas tem depressão segundo o PHQ.
* **Equilíbrio Geral (F1-Score = 0.62; Acurácia = 81,3%):** O F1-score da classe positiva é 0.62 e o modelo acerta 81,3% das classificações em uma população na qual apenas 20,4% têm depressão (9.894.699 casos reais).
* **Conclusão Comparativa:** Com o mesmo split e as mesmas variáveis, o SVM é um pouco mais sensível que o XGBoost (Recall de 75,4% vs. 72,6%, ou cerca de 275 mil casos a mais detectados), mas gera cerca de 850 mil falsos positivos a mais e tem acurácia, precisão e F1 ligeiramente menores. Como o ROC-AUC é praticamente igual (0.8703 vs. 0.8718), a escolha depende da prioridade do rastreio: o SVM é preferível para **minimizar casos não detectados**; o XGBoost, para **reduzir falsos alarmes**.

### Benchmark Comparativo dos Modelos Binários (sem variáveis de depressão e ansiedade em $X$)

| Métrica Populacional | Random Forest (Otimizado)† | XGBoost (Otimizado) | SVM (LinearSVC) | Destaque Prático |
| :--- | :---: | :---: | :---: | :--- |
| **ROC-AUC Score** | pendente | **0.8718** | 0.8703 | Praticamente empatados (diferença de 0.0015) |
| **Acurácia Global** | pendente | **82.52%** | 81.33% | **XGBoost acerta mais no geral** |
| **Precisão (Classe 1)** | pendente | **55.53%** | 53.05% | **XGBoost gera menos falsos alarmes** |
| **Recall / Sensibilidade** | pendente | 72.57% | **75.36%** | **SVM captura mais casos reais** |
| **F1-Score (Classe 1)** | pendente | **0.63** | 0.62 | XGBoost obtém um equilíbrio ligeiramente melhor |

† **Random Forest pendente:** os valores anteriores (ROC-AUC 0.9135, acurácia 86%, precisão 67%, recall 71.76% e F1 0.70) foram obtidos com as variáveis de ansiedade ainda em $X$ (`03 Modelos supervisionados/Random Forest/06_02_preparacao_classificacao_2025.ipynb`) e **não são comparáveis** com as colunas de XGBoost e SVM. Reexecute o Random Forest com a mesma lista `COLUNAS_SINTOMAS` para preencher esta coluna.